<h3> Lecture de la base géographique nationale du bâtiment (BDNB) et exploration de ses différentes couches (layers) et de leurs colonnes. </h3>

https://open-data.s3.fr-par.scw.cloud/bdnb_millesime_2025-07-a/millesime_2025-07-a_france/open_data_millesime_2025-07-a_france_gpkg.tar.gz

In [1]:
gpkg_path = "C:\\Users\\DELL\\Desktop\\Data4Good local\\PCC\\open_data_millesime_2025-07-a_france_gpkg\\gpkg\\bdnb.gpkg"

import duckdb
import sqlite3

con = duckdb.connect()

# Connexion en lecture seule à la base GeoPackage
con_sqlite = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
cursor = con_sqlite.cursor()

try:
    con.execute("INSTALL spatial")
except Exception:
    pass

con.execute("LOAD spatial")

con.close()

Liste des tables de la db BDNB

In [ ]:
# Liste des layers vectoriels (gpkg_contents table_name)
cursor.execute("""
    SELECT table_name
    FROM gpkg_contents
    WHERE data_type = 'features'
""")
layers = [row[0] for row in cursor.fetchall()]

for layer in layers:
    print(layer)

adresse_compile
rel_batiment_construction_adresse
rel_batiment_groupe_adresse
batiment_construction
batiment_groupe_compile
parcelle
rel_batiment_construction_rnb


In [ ]:
'''
for layer in layers:
    # Lire seulement les métadonnées ou un aperçu
    print(con.execute(
        f"SELECT * FROM ST_Read('{gpkg_path}', layer => '{layer}', keep_wkb => TRUE) LIMIT 1"
    ).fetchall())'''

'\nfor layer in layers:\n    # Lire seulement les métadonnées ou un aperçu\n    print(con.execute(\n        f"SELECT * FROM ST_Read(\'{gpkg_path}\', layer => \'{layer}\', keep_wkb => TRUE) LIMIT 1"\n    ).fetchall())'

Liste des noms de colonnes de chaque table de la db BDNB

In [ ]:
for layer in layers:
    print(f"\nLayer: {layer}")

    # Lire le schéma de la table
    cursor.execute(f'PRAGMA table_info("{layer}")')
    columns = cursor.fetchall()

    for col in columns:
        # col = (cid, name, type, notnull, dflt_value, pk)
        print(f"  - {col[1]} ({col[2]})")


Layer: adresse_compile
  - fid (INTEGER)
  - geom_adresse (GEOMETRY)
  - cle_interop_adr (TEXT)
  - code_departement_insee (TEXT(2))
  - numero (MEDIUMINT)
  - rep (TEXT)
  - type_voie (TEXT)
  - nom_voie (TEXT)
  - libelle_adresse (TEXT)
  - code_postal (TEXT(5))
  - code_commune_insee (TEXT(5))
  - libelle_commune (TEXT)
  - source (TEXT)
  - nb_bat_grp (MEDIUMINT)
  - nb_bat_grp_hors_dep (MEDIUMINT)

Layer: rel_batiment_construction_adresse
  - fid (INTEGER)
  - geom_batiment_construction_adresse (GEOMETRY)
  - batiment_construction_id (TEXT)
  - cle_interop_adr (TEXT)
  - code_departement_insee (TEXT(2))
  - distance_batiment_construction_adresse (MEDIUMINT)
  - fiabilite (MEDIUMINT)
  - adresse_principale (BOOLEAN)

Layer: rel_batiment_groupe_adresse
  - fid (INTEGER)
  - geom_bat_adresse (GEOMETRY)
  - batiment_groupe_id (TEXT(22))
  - cle_interop_adr (TEXT)
  - code_departement_insee (TEXT(2))
  - origine (TEXT)
  - fiabilite (MEDIUMINT)

Layer: batiment_construction
  - fid (I

Fonction d'exploration de chaque table : nombre de lignes, head

In [ ]:
def explore_layer(layer_name):
    con_sqlite = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
    cursor = con_sqlite.cursor()

    cursor.execute("""
        SELECT feature_count
        FROM gpkg_ogr_contents
        WHERE table_name = ?
    """, (layer_name,))

    row = cursor.fetchone()

    if row:
        print(f"Nombre de features dans {layer_name}: {format(row[0], ',')}")
    else:
        print("Feature count non disponible dans les métadonnées")

    result = con.execute(f"""
        SELECT 
            *
        FROM ST_Read('{gpkg_path}', layer => '{layer_name}')
        LIMIT 5
    """).fetchdf()

    con_sqlite.close()
    display(result)



In [ ]:
for layer in layers:
    explore_layer(layer)

Nombre de features dans adresse_compile: 26,348,056


,cle_interop_adr,code_departement_insee,numero,rep,type_voie,nom_voie,libelle_adresse,code_postal,code_commune_insee,libelle_commune,source,nb_bat_grp,nb_bat_grp_hors_dep,geom_adresse
0,0010000A093100101,01,56,None,Imp,Des Lilas,56.0 IMP DES LILAS L'Abergement-Clémenciat,None,01001,L'Abergement-Clémenciat,Arcep,<NA>,<NA>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,0010000A095000101,01,89,None,Imp,Des Chenes,89.0 IMP DES CHENES L'Abergement-Clémenciat,None,01001,L'Abergement-Clémenciat,Arcep,<NA>,<NA>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,0010000A095100101,01,133,None,Imp,Des Chenes,133.0 IMP DES CHENES L'Abergement-Clémenciat,None,01001,L'Abergement-Clémenciat,Arcep,<NA>,<NA>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,0010000A095200101,01,132,None,Imp,Des Chenes,132.0 IMP DES CHENES L'Abergement-Clémenciat,None,01001,L'Abergement-Clémenciat,Arcep,<NA>,<NA>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,0010000A095400101,01,36,None,Imp,Des Chenes,36.0 IMP DES CHENES L'Abergement-Clémenciat,None,01001,L'Abergement-Clémenciat,Arcep,<NA>,<NA>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


Nombre de features dans rel_batiment_construction_adresse: 33,165,668


,batiment_construction_id,cle_interop_adr,code_departement_insee,distance_batiment_construction_adresse,fiabilite,adresse_principale,geom_batiment_construction_adresse
0,bdnb-bc-MRYF-3PQ7-79P4,01002_0100_00033,01,21,9,False,"[4, 4, 0, 0, 0, 0, 0, 0, 175, 166, 88, 73, 54,..."
1,bdnb-bc-JLG9-32GQ-M1W5,01002_0200_00070,01,12,17,True,"[4, 4, 0, 0, 0, 0, 0, 0, 157, 172, 88, 73, 9, ..."
2,bdnb-bc-JLG9-32GQ-M1W5,01002_0200_00110,01,24,9,True,"[4, 4, 0, 0, 0, 0, 0, 0, 13, 173, 88, 73, 8, 2..."
3,bdnb-bc-HJAU-QBZ6-XK1H,01004_0234_00034,01,18,9,False,"[4, 4, 0, 0, 0, 0, 0, 0, 14, 90, 87, 73, 32, 1..."
4,bdnb-bc-AKYR-A6BN-RTDP,01004_0430_00001,01,38,9,False,"[4, 4, 0, 0, 0, 0, 0, 0, 53, 250, 86, 73, 128,..."


Nombre de features dans rel_batiment_groupe_adresse: 27,432,117


...

,batiment_groupe_id,code_departement_insee,s_geom_groupe,code_iris,code_commune_insee,libelle_commune_insee,code_epci_insee,contient_fictive_geom_groupe,cle_interop_adr_principale_ban,libelle_adr_principale_ban,...,rpls_s_log_hab,rpls_type_construction,usage_principal_bdnb_open,contrainte_urbanisme_ac1,zone_plu_bati_patrimonial,monument_historique,source_monument_historique,denomination_monument_historique,distance_monument_historique,geom_groupe
0,bdnb-bg-1111-1HRT-8SVH,70,266,701380000,70138,Chassey-lès-Scey,247000367,False,70138_0025_00007,7 Rue de la Vierge 70360 Chassey-lès-Scey,...,NaN,None,Résidentiel individuel,<NA>,<NA>,<NA>,None,None,<NA>,"[5, 4, 0, 0, 0, 0, 0, 0, 211, 129, 97, 73, 2, ..."
1,bdnb-bg-1111-2AYZ-DTTT,29,78,292690000,29269,Saint-Thurien,242900694,False,29269_xd3yiu_00005,5 Lieu dit Kerbihan 29380 Saint-Thurien,...,NaN,None,Résidentiel individuel,<NA>,<NA>,<NA>,None,None,<NA>,"[5, 4, 0, 0, 0, 0, 0, 0, 108, 111, 75, 72, 14,..."
2,bdnb-bg-1111-3Q5V-HLV3,14,127,142810000,14281,Formigny La Bataille,200066801,False,14281_0040_00077,77 RUE CAPET 14710 Formigny La Bataille,...,NaN,None,Résidentiel individuel,True,False,False,None,None,<NA>,"[5, 4, 0, 0, 0, 0, 0, 0, 201, 113, 203, 72, 24..."
3,bdnb-bg-1111-5MR8-UMTV,80,15,804950000,80495,Lucheux,200070951,False,None,None,...,NaN,None,None,True,False,False,merimee + geoportail_urbanisme,Eglise,362,"[5, 4, 0, 0, 0, 0, 0, 0, 6, 115, 32, 73, 151, ..."
4,bdnb-bg-1111-5Q2T-324X,62,11,627040000,62704,Renty,200069037,False,None,None,...,NaN,None,None,<NA>,<NA>,<NA>,None,None,<NA>,"[5, 4, 0, 0, 0, 0, 0, 0, 235, 229, 26, 73, 125..."


Nombre de features dans parcelle: 93,303,788


,parcelle_id,code_departement_insee,code_commune_insee,s_geom_parcelle,geom_parcelle
0,010010000A0001,01,01001,14394.0,"[5, 4, 0, 0, 0, 0, 0, 0, 168, 246, 78, 73, 27,..."
1,010010000A0002,01,01001,666.0,"[5, 4, 0, 0, 0, 0, 0, 0, 235, 255, 78, 73, 55,..."
2,010010000A0003,01,01001,1411.0,"[5, 4, 0, 0, 0, 0, 0, 0, 238, 1, 79, 73, 25, 1..."
3,010010000A0005,01,01001,797.0,"[5, 4, 0, 0, 0, 0, 0, 0, 41, 9, 79, 73, 76, 10..."
4,010010000A0006,01,01001,53723.0,"[5, 4, 0, 0, 0, 0, 0, 0, 251, 250, 78, 73, 75,..."


Nombre de features dans rel_batiment_construction_rnb: 41,877,354


,rnb_id,batiment_construction_id,code_departement_insee,type_appariement,detail,geom_rnb_pos
0,11111MR38XH7,bdnb-bc-U61J-N4N9-52UC,37,"Alignement 1 BC = 1 RNB, recouvrement géométri...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,11112X4VWBDF,bdnb-bc-W5F2-GYLN-67V2,41,"Alignement 1 BC = 1 RNB, recouvrement géométri...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,11115G39BK9D,bdnb-bc-XRXT-AMYS-U1TY,33,"Alignement 1 BC = 1 RNB, recouvrement géométri...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,1111737XMGFG,bdnb-bc-9NKQ-M5EQ-7PWY,18,"Alignement 1 BC = 1 RNB, recouvrement géométri...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,11117H922PGK,bdnb-bc-HUEJ-M779-57F9,37,"Alignement 1 BC = 1 RNB, recouvrement géométri...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


### Exploration de la table contenant le plus de données : batiment_groupe_compile (277 colonnes x 32 220 04 lignes)

In [ ]:
layer_name = 'batiment_groupe_compile'

conn = sqlite3.connect(f"file:{gpkg_path}?mode=ro", uri=True)
cursor = conn.cursor()


print(f"\nLayer: {layer}")

# Lire le schéma de la table
cursor.execute(f'PRAGMA table_info("{layer_name}")')
columns = cursor.fetchall()

for col in columns:
    # col = (cid, name, type, notnull, dflt_value, pk)
    print(f"  - {col[1]} ({col[2]})")

conn.close()



Layer: batiment_groupe_compile
  - fid (INTEGER)
  - geom_groupe (GEOMETRY)
  - batiment_groupe_id (TEXT(22))
  - code_departement_insee (TEXT(2))
  - s_geom_groupe (MEDIUMINT)
  - code_iris (TEXT(9))
  - code_commune_insee (TEXT(5))
  - libelle_commune_insee (TEXT)
  - code_epci_insee (TEXT)
  - contient_fictive_geom_groupe (BOOLEAN)
  - cle_interop_adr_principale_ban (TEXT)
  - libelle_adr_principale_ban (TEXT)
  - nb_adresse_valid_ban (MEDIUMINT)
  - fiabilite_cr_adr_niv_1 (TEXT)
  - fiabilite_cr_adr_niv_2 (TEXT)
  - bdtopo_bat_l_nature (TEXT)
  - bdtopo_bat_l_usage_1 (TEXT)
  - bdtopo_bat_l_usage_2 (TEXT)
  - bdtopo_bat_l_etat (TEXT)
  - bdtopo_bat_hauteur_mean (MEDIUMINT)
  - bdtopo_bat_max_hauteur (MEDIUMINT)
  - bdtopo_bat_altitude_sol_mean (MEDIUMINT)
  - bdtopo_equ_l_nature (TEXT)
  - bdtopo_equ_l_nature_detaillee (TEXT)
  - bdtopo_equ_l_toponyme (TEXT)
  - bdtopo_zoa_l_nature (TEXT)
  - bdtopo_zoa_l_nature_detaillee (TEXT)
  - bdtopo_zoa_l_toponyme (TEXT)
  - bpe_l_type_equi

In [ ]:
cols_to_display = ['batiment_groupe_id','geom_groupe']
cols_to_display += [col[1] for col in columns if col[1].startswith("ffo_")]
cols_to_display += [ 'code_commune_insee ','code_departement_insee', 'batiment_groupe_id', 'croisement_geospx_reussi', 'alea_argile', 'rnc_l_annee_construction', 'rnc_nb_log', 'rnc_nb_lot_tertiaire', 'fiabilite_emprise_sol']
print(', '.join(cols_to_display)) 

result = con.execute(f"""
    SELECT 
        {', '.join(cols_to_display)}
    FROM ST_Read('{gpkg_path}', layer => '{layer_name}')
    LIMIT 20
""").fetchdf()

display(result)



NameError: name 'columns' is not defined

Création d'un fichier centroids.parquet (7') pour ne garder que les colonnes suivantes : commune, annee_construction, rga et les coordonnées géographiques sous forme de centroïde de chaque bâtiment.

In [5]:

# Configuration
layer = "batiment_groupe_compile"

centroids_tb_cols = [
    'batiment_groupe_id', 'code_commune_insee', 'code_departement_insee',
    'ffo_bat_annee_construction', 'ffo_bat_usage_niveau_1_txt', 'ffo_bat_nb_log'
]

# 1. Créer centroids.parquet avec id_row
con = duckdb.connect()
con.execute("INSTALL spatial")
con.execute("LOAD spatial")

# Ajouter id_row et exporter en Parquet
cols_sql = ", ".join(centroids_tb_cols)
con.execute(f"""
COPY (
    SELECT
        {cols_sql},
        alea_argile as alea_argile_old,
        ST_Centroid(geom_groupe) AS centroid,
        row_number() OVER () AS id_row  -- Ajout de la colonne id_row
    FROM ST_Read('{gpkg_path}', layer => '{layer}')
)
TO 'centroids.parquet'
(FORMAT PARQUET)
""")
con.close()

# Mettre à jour la liste des colonnes
centroids_tb_cols += ['alea_argile_old', 'id_row']

Export des données rga strictement utiles en parquet pour croisement avec les données de la bdnb

In [ ]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL spatial")
con.execute("LOAD spatial")
con.execute("PRAGMA threads=16")

con.execute("""
COPY (
SELECT niveau, geom, insee_dep
FROM ST_Read('../dbt_pipeline/pipeline_inputs/AleaRG_2025_Fxx_L93.gpkg')
)
TO 'alea_rga.parquet' (FORMAT PARQUET);
""")

<h3>Travail sur les centroides des bâtiments (centroids.parquet)</h3>

Croisement des centroids avec la table rga juillet 2026 pour affecter le niveau de risque rga à chaque bâtiment

10 heures

In [5]:
'''import duckdb
import geopandas as gpd

# Configuration
centroids_tb_cols = ['ffo_bat_annee_construction', 'code_commune_insee', 'code_departement_insee',
              'ffo_bat_usage_niveau_1_txt', 'ffo_bat_nb_log', 'alea_argile_old']
gp_centroids = "centroids.parquet"
gp_rga       = "alea_rga.parquet"
out_file     = "centroids_rga.parquet"
batch_size   = 50000  # Taille optimale pour 32 Go de RAM

# Connexion et configuration
con = duckdb.connect()
con.execute("INSTALL spatial")
con.execute("LOAD spatial")
con.execute("PRAGMA threads=8")  # 8 threads pour ton CPU
con.execute("PRAGMA memory_limit='32GB'")

# Préparer la table RGA avec bbox et index spatial
con.execute(f"""
CREATE OR REPLACE TABLE rga AS
SELECT
    insee_dep,
    niveau,
    geom,
    ST_XMin(geom) AS minx,
    ST_XMax(geom) AS maxx,
    ST_YMin(geom) AS miny,
    ST_YMax(geom) AS maxy
FROM '{gp_rga}'
""")
con.execute("CREATE INDEX idx_rga_geom ON rga USING RTree(geom)")

# Créer la table results avec toutes les colonnes nécessaires + rga_niveau
con.execute(f"""
CREATE OR REPLACE TABLE results AS
SELECT
    {", ".join(centroids_tb_cols)},
    NULL AS rga_niveau  -- Ajout de la colonne rga_niveau
FROM '{gp_centroids}'
LIMIT 0
""")

# Récupérer les IDs min et max pour la pagination
min_id = con.execute(f"SELECT MIN(id_row) FROM '{gp_centroids}'").fetchone()[0]
max_id = con.execute(f"SELECT MAX(id_row) FROM '{gp_centroids}'").fetchone()[0]
batches = ((max_id - min_id) // batch_size) + 1

# Traitement par batches
current_min_id = min_id
for i in range(batches):  # Remets la boucle complète pour traiter tous les batches
    current_max_id = current_min_id + batch_size - 1
    print(f"Traitement batch {i+1}/{batches} (ID {current_min_id} à {current_max_id})")

    con.execute(f"""
    INSERT INTO results
    WITH c_batch AS MATERIALIZED (
        SELECT *
        FROM '{gp_centroids}'
        WHERE id_row BETWEEN {current_min_id} AND {current_max_id}
    ),
    joined AS MATERIALIZED (
        SELECT
            c_batch.*,
            r.niveau AS rga_niveau  -- On conserve directement rga_niveau
        FROM c_batch
        LEFT JOIN rga r
        ON c_batch.code_departement_insee = r.insee_dep
        AND ST_X(c_batch.centroid) BETWEEN r.minx AND r.maxx
        AND ST_Y(c_batch.centroid) BETWEEN r.miny AND r.maxy
        AND ST_Intersects(c_batch.centroid, r.geom)
    )
    SELECT DISTINCT
        {", ".join(centroids_tb_cols)},
        rga_niveau  -- On inclut directement rga_niveau dans les résultats
    FROM joined
    """)

    current_min_id = current_max_id + 1

# Écrire le résultat final en une seule fois
con.execute(f"""
COPY (
    SELECT * FROM results
) TO '{out_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.close()'''


Traitement batch 1/645 (ID 1 à 50000)
Traitement batch 2/645 (ID 50001 à 100000)
Traitement batch 3/645 (ID 100001 à 150000)
Traitement batch 4/645 (ID 150001 à 200000)
Traitement batch 5/645 (ID 200001 à 250000)
Traitement batch 6/645 (ID 250001 à 300000)
Traitement batch 7/645 (ID 300001 à 350000)
Traitement batch 8/645 (ID 350001 à 400000)
Traitement batch 9/645 (ID 400001 à 450000)
Traitement batch 10/645 (ID 450001 à 500000)
Traitement batch 11/645 (ID 500001 à 550000)
Traitement batch 12/645 (ID 550001 à 600000)
Traitement batch 13/645 (ID 600001 à 650000)
Traitement batch 14/645 (ID 650001 à 700000)
Traitement batch 15/645 (ID 700001 à 750000)
Traitement batch 16/645 (ID 750001 à 800000)
Traitement batch 17/645 (ID 800001 à 850000)
Traitement batch 18/645 (ID 850001 à 900000)
Traitement batch 19/645 (ID 900001 à 950000)
Traitement batch 20/645 (ID 950001 à 1000000)
Traitement batch 21/645 (ID 1000001 à 1050000)
Traitement batch 22/645 (ID 1050001 à 1100000)
Traitement batch 23/6

In [9]:
import pandas as pd

df = pd.read_parquet(out_file)

df

,ffo_bat_annee_construction,code_commune_insee,code_departement_insee,ffo_bat_usage_niveau_1_txt,ffo_bat_nb_log,alea_argile_old,rga_niveau
0,NaN,67046,67,None,NaN,Faible,1.0
1,NaN,67098,67,None,NaN,Faible,1.0
2,1956.0,67506,67,Résidentiel individuel,1.0,Faible,1.0
3,1977.0,88094,88,Résidentiel individuel,1.0,Faible,1.0
4,NaN,63397,63,None,NaN,Moyen,3.0
...,...,...,...,...,...,...,...
45092,1986.0,77317,77,Résidentiel individuel,1.0,Fort,3.0
45093,1988.0,84016,84,Résidentiel individuel,1.0,Moyen,2.0
45094,1963.0,11266,11,Tertiaire & Autres,1.0,Moyen,2.0
45095,1963.0,11262,11,Résidentiel individuel,1.0,Moyen,2.0


Agrégation du nombre de bâtiment par date de construction et par niveau de risque

In [ ]:
con = duckdb.connect()
df_centroids = con.execute("SELECT * FROM 'centroids_rga.parquet'").fetchdf()

con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='32GB'")

con.execute("""
COPY (

SELECT
    code_commune_insee,
    COUNT(*) AS nb_batiments,

    SUM(ffo_bat_annee_construction BETWEEN 1945 AND 1975 AND rga_niveau=1)::INTEGER AS rga_1945_1975_faible,
    SUM(ffo_bat_annee_construction BETWEEN 1945 AND 1975 AND rga_niveau=2)::INTEGER  AS rga_1945_1975_moyen,
    SUM(ffo_bat_annee_construction BETWEEN 1945 AND 1975 AND rga_niveau is null)::INTEGER    AS rga_1945_1975_nul,
    SUM(ffo_bat_annee_construction BETWEEN 1945 AND 1975 AND rga_niveau=3)::INTEGER   AS rga_1945_1975_fort,

    SUM(ffo_bat_annee_construction BETWEEN 1976 AND 2020 AND rga_niveau=1)::INTEGER AS rga_1976_2020_faible,
    SUM(ffo_bat_annee_construction BETWEEN 1976 AND 2020 AND rga_niveau=2)::INTEGER  AS rga_1976_2020_moyen,
    SUM(ffo_bat_annee_construction BETWEEN 1976 AND 2020 AND rga_niveau is null)::INTEGER    AS rga_1976_2020_nul,
    SUM(ffo_bat_annee_construction BETWEEN 1976 AND 2020 AND rga_niveau=3)::INTEGER   AS rga_1976_2020_fort,

    SUM(ffo_bat_annee_construction > 2020 AND rga_niveau=1)::INTEGER AS rga_post2020_faible,
    SUM(ffo_bat_annee_construction > 2020 AND rga_niveau=2)::INTEGER  AS rga_post2020_moyen,
    SUM(ffo_bat_annee_construction > 2020 AND rga_niveau is null)::INTEGER    AS rga_post2020_nul,
    SUM(ffo_bat_annee_construction > 2020 AND rga_niveau=3)::INTEGER   AS rga_post2020_fort,

    SUM(ffo_bat_annee_construction < 1945 AND rga_niveau=1)::INTEGER AS rga_pre1945_faible,
    SUM(ffo_bat_annee_construction < 1945 AND rga_niveau=2)::INTEGER  AS rga_pre1945_moyen,
    SUM(ffo_bat_annee_construction < 1945 AND rga_niveau is null)::INTEGER    AS rga_pre1945_nul,
    SUM(ffo_bat_annee_construction < 1945 AND rga_niveau=3)::INTEGER   AS rga_pre1945_fort,

    SUM(ffo_bat_annee_construction IS NULL AND rga_niveau=1)::INTEGER AS rga_unk_faible,
    SUM(ffo_bat_annee_construction IS NULL AND rga_niveau=2)::INTEGER  AS rga_unk_moyen,
    SUM(ffo_bat_annee_construction IS NULL AND rga_niveau is null)::INTEGER    AS rga_unk_nul,
    SUM(ffo_bat_annee_construction IS NULL AND rga_niveau=3)::INTEGER   AS rga_unk_fort

FROM 'centroids_rga.parquet'

GROUP BY code_commune_insee

)
TO 'agregation_communes_rga.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

<h3>Exploration des des données RGA / date de construction des communes (nombre de bâtiments concernés)</h3>

In [40]:
import pandas as pd

agregation_df = pd.read_csv('../dbt_pipeline/pipeline_inputs/communes_rga_2607.csv')
display(agregation_df)

,code_commune_insee,nb_batiments,rga_1945_1975_faible,rga_1945_1975_moyen,rga_1945_1975_nul,rga_1945_1975_fort,rga_1976_2020_faible,rga_1976_2020_moyen,rga_1976_2020_nul,rga_1976_2020_fort,...,rga_post2020_nul,rga_post2020_fort,rga_pre1945_faible,rga_pre1945_moyen,rga_pre1945_nul,rga_pre1945_fort,rga_unk_faible,rga_unk_moyen,rga_unk_nul,rga_unk_fort
0,52529,951,0.0,46.0,1.0,28.0,0.0,164.0,1.0,39.0,...,0.0,5.0,1.0,309.0,1.0,25.0,0.0,260.0,4,56.0
1,45244,1858,0.0,252.0,0.0,43.0,0.0,652.0,0.0,100.0,...,0.0,2.0,0.0,250.0,0.0,33.0,0.0,431.0,0,86.0
2,35279,766,11.0,1.0,29.0,0.0,120.0,48.0,151.0,0.0,...,13.0,0.0,82.0,16.0,75.0,0.0,68.0,25.0,87,0.0
3,77083,3590,0.0,287.0,0.0,379.0,0.0,561.0,0.0,987.0,...,0.0,54.0,0.0,241.0,0.0,296.0,0.0,362.0,0,412.0
4,59574,5717,11.0,345.0,0.0,969.0,7.0,394.0,0.0,550.0,...,0.0,14.0,182.0,411.0,0.0,1653.0,101.0,346.0,0,730.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34781,31593,99,0.0,0.0,0.0,3.0,0.0,0.0,0.0,24.0,...,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0,47.0
34782,52291,90,5.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,...,0.0,0.0,37.0,0.0,0.0,0.0,41.0,0.0,0,0.0
34783,32008,72,0.0,0.0,0.0,1.0,0.0,0.0,0.0,13.0,...,0.0,2.0,0.0,0.0,0.0,27.0,0.0,0.0,0,29.0
34784,26351,45,0.0,3.0,0.0,0.0,0.0,8.0,0.0,0.0,...,0.0,0.0,0.0,13.0,0.0,0.0,0.0,19.0,1,0.0


In [ ]:
df = agregation_df

dfs = pd.DataFrame()

dfs.loc['pre1945', 'rga_nul'] = df.loc[:, 'rga_pre1945_nul'].sum()
dfs.loc['pre1945', 'rga_faible'] = df.loc[:, 'rga_pre1945_faible'].sum()
dfs.loc['pre1945', 'rga_moyen'] = df.loc[:, 'rga_pre1945_moyen'].sum()
dfs.loc['pre1945', 'rga_fort'] = df.loc[:, 'rga_pre1945_fort'].sum()
dfs.loc['1945-1975', 'rga_nul'] = df.loc[:, 'rga_1945_1975_nul'].sum()
dfs.loc['1945-1975', 'rga_faible'] = df.loc[:, 'rga_1945_1975_faible'].sum()
dfs.loc['1945-1975', 'rga_moyen'] = df.loc[:, 'rga_1945_1975_moyen'].sum()
dfs.loc['1945-1975', 'rga_fort'] = df.loc[:, 'rga_1945_1975_fort'].sum()
dfs.loc['1976-2020', 'rga_nul'] = df.loc[:, 'rga_1976_2020_nul'].sum()
dfs.loc['1976-2020', 'rga_faible'] = df.loc[:, 'rga_1976_2020_faible'].sum()
dfs.loc['1976-2020', 'rga_moyen'] = df.loc[:, 'rga_1976_2020_moyen'].sum()
dfs.loc['1976-2020', 'rga_fort'] = df.loc[:, 'rga_1976_2020_fort'].sum()
dfs.loc['post2020', 'rga_nul'] = df.loc[:, 'rga_post2020_nul'].sum()
dfs.loc['post2020', 'rga_faible'] = df.loc[:, 'rga_post2020_faible'].sum()
dfs.loc['post2020', 'rga_moyen'] = df.loc[:, 'rga_post2020_moyen'].sum()
dfs.loc['post2020', 'rga_fort'] = df.loc[:, 'rga_post2020_fort'].sum()
dfs.loc['unk', 'rga_nul'] = df.loc[:, 'rga_unk_nul'].sum()
dfs.loc['unk', 'rga_faible'] = df.loc[:, 'rga_unk_faible'].sum()
dfs.loc['unk', 'rga_moyen'] = df.loc[:, 'rga_unk_moyen'].sum()
dfs.loc['unk', 'rga_fort'] = df.loc[:, 'rga_unk_fort'].sum()    

df_tot = dfs.copy()
df_tot['Total'] = df_tot.sum(axis=1) 
df_tot.loc['Total'] = df_tot.sum(axis=0) 

df_tot.map(lambda x: f"{x:,.0f}".replace(",", " ") if isinstance(x, (int, float)) else x)



,rga_nul,rga_faible,rga_moyen,rga_fort,Total
pre1945,1 396 601,1 449 991,3 053 121,1 429 635,7 329 348
1945-1975,707 255,808 689,1 595 705,760 500,3 872 149
1976-2020,1 603 038,1 873 041,3 751 410,1 958 596,9 186 085
post2020,82 801,94 245,179 927,93 186,450 159
unk,1 607 076,1 632 030,3 304 299,1 698 887,8 242 292
Total,5 396 771,5 857 996,11 884 462,5 940 804,29 080 033


In [38]:
print("Nombre de bâtiments construits avant 1945 : {:,.0f}".format(dfs.loc['pre1945'].sum()).replace(",", " "))

print("Nombre de bâtiments construits entre 1945 et 1975 : {:,.0f}".format(dfs.loc['1945-1975'].sum()).replace(",", " "))

print("Nombre de bâtiments construits entre 1976 et 2020 : {:,.0f}".format(dfs.loc['1976-2020'].sum()).replace(",", " "))

print("Nombre de bâtiments construits après 2020 : {:,.0f}".format(dfs.loc['post2020'].sum()).replace(",", " "))

print("Nombre de bâtiments de date de construction connue : {:,.0f}".format(dfs.loc['pre1945'].sum()+ dfs.loc['1945-1975'].sum() + dfs.loc['1976-2020'].sum() + dfs.loc['post2020'].sum()).replace(",", " "))

print("Nombre de bâtiments de date de construction inconnue : {:,.0f}".format(dfs.loc['unk'].sum()).replace(",", " "))



Nombre de bâtiments construits avant 1945 : 7 329 348
Nombre de bâtiments construits entre 1945 et 1975 : 3 872 149
Nombre de bâtiments construits entre 1976 et 2020 : 9 186 085
Nombre de bâtiments construits après 2020 : 450 159
Nombre de bâtiments de date de construction connue : 20 837 741
Nombre de bâtiments de date de construction inconnue : 8 242 292
